# Rerun Visualization: CPU MuJoCo vs MJX-Warp GPU Contacts

This notebook uses the Rerun SDK (`import rerun as rr`) to visualize the CPU and GPU contact data in 3D.

It logs:

- CPU and GPU ball centers as animated `Points3D`
- CPU and GPU contact points as animated `Points3D`
- contact points colored by distance to the corresponding hand collision mesh
- all historical contact points as a very low-opacity background cloud
- non-contacting balls with very low opacity and current-frame contacting balls highlighted
- palm trajectories as `LineStrips3D`
- animated MANO hand URDF meshes as `Mesh3D`

The CPU and GPU scenes are shown side by side with a fixed X offset. The original coordinates are preserved up to that visualization offset.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any
import uuid
import xml.etree.ElementTree as ET

import mujoco
import numpy as np
import pandas as pd
import rerun as rr
import trimesh

print('rerun', rr.__version__)
print('mujoco', mujoco.__version__)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'assets/mano_hand_s02').exists() and (candidate / 'docker/mujoco_mjx').exists():
            return candidate
    raise RuntimeError('Run this notebook from the standalone mujoco-warp-contactbench repo, or set ROOT manually.')

ROOT = find_repo_root()
LOGS = ROOT / 'logs'
HAND_XML = ROOT / 'assets/mano_hand_s02/mjcf/mano_hand_s02_full_convex.xml'
HAND_URDF = ROOT / 'assets/mano_hand_s02/urdf/mano_hand_s02_full_convex.urdf'

def resolve_data_file(name: str) -> Path:
    generated = LOGS / name
    if generated.exists():
        return generated
    raise FileNotFoundError(f'Missing logs/{name}; generate JSON debug outputs before using this notebook view.')

CPU_JSON = resolve_data_file('mujoco_cpu_ball_pit_contact_10s.json')
GPU_JSON = resolve_data_file('mjx_warp_gpu_ball_pit_contact_10s.json')
GPU_NATIVE_JSON = resolve_data_file('mjx_warp_gpu_ball_pit_contact_native_10s.json')

for path in [HAND_URDF, CPU_JSON, GPU_JSON, GPU_NATIVE_JSON]:
    status = 'ok' if path.exists() else 'missing'
    print(f'{path.relative_to(ROOT)}: {status}')


In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

cpu_payload = read_json(CPU_JSON)
gpu_payload = read_json(GPU_JSON)
gpu_native_payload = read_json(GPU_NATIVE_JSON)

print('CPU frames:', len(cpu_payload['contact']))
print('GPU frames:', len(gpu_payload['contact']))
print('CPU objects:', len(cpu_payload['object_trajectories']))
print('GPU objects:', len(gpu_payload['object_trajectories']))


## Geometry Helpers

MuJoCo compiles STL meshes with non-zero `geom_pos/geom_quat`. We transform the compiled mesh vertices into each hand link body frame before computing contact-point distance.


In [ ]:
def quat_to_mat(q: np.ndarray) -> np.ndarray:
    w, x, y, z = [float(v) for v in q]
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w, 2*x*z + 2*y*w],
        [2*x*y + 2*z*w, 1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w, 2*y*z + 2*x*w, 1 - 2*x*x - 2*y*y],
    ], dtype=np.float64)

def point_triangle_dist2(points: np.ndarray, tri: np.ndarray) -> np.ndarray:
    p = points
    a, b, c = tri[0], tri[1], tri[2]
    ab = b - a
    ac = c - a
    ap = p - a
    d1 = ap @ ab
    d2 = ap @ ac
    out = np.empty((p.shape[0],), dtype=np.float64)
    done = np.zeros((p.shape[0],), dtype=bool)

    mask = (d1 <= 0.0) & (d2 <= 0.0)
    out[mask] = np.sum((p[mask] - a) ** 2, axis=1)
    done |= mask

    bp = p - b
    d3 = bp @ ab
    d4 = bp @ ac
    mask = (~done) & (d3 >= 0.0) & (d4 <= d3)
    out[mask] = np.sum((p[mask] - b) ** 2, axis=1)
    done |= mask

    vc = d1 * d4 - d3 * d2
    mask = (~done) & (vc <= 0.0) & (d1 >= 0.0) & (d3 <= 0.0)
    if np.any(mask):
        v = d1[mask] / (d1[mask] - d3[mask])
        proj = a + v[:, None] * ab
        out[mask] = np.sum((p[mask] - proj) ** 2, axis=1)
    done |= mask

    cp = p - c
    d5 = cp @ ab
    d6 = cp @ ac
    mask = (~done) & (d6 >= 0.0) & (d5 <= d6)
    out[mask] = np.sum((p[mask] - c) ** 2, axis=1)
    done |= mask

    vb = d5 * d2 - d1 * d6
    mask = (~done) & (vb <= 0.0) & (d2 >= 0.0) & (d6 <= 0.0)
    if np.any(mask):
        w = d2[mask] / (d2[mask] - d6[mask])
        proj = a + w[:, None] * ac
        out[mask] = np.sum((p[mask] - proj) ** 2, axis=1)
    done |= mask

    va = d3 * d6 - d5 * d4
    mask = (~done) & (va <= 0.0) & ((d4 - d3) >= 0.0) & ((d5 - d6) >= 0.0)
    if np.any(mask):
        w = (d4[mask] - d3[mask]) / ((d4[mask] - d3[mask]) + (d5[mask] - d6[mask]))
        proj = b + w[:, None] * (c - b)
        out[mask] = np.sum((p[mask] - proj) ** 2, axis=1)
    done |= mask

    mask = ~done
    if np.any(mask):
        denom = va[mask] + vb[mask] + vc[mask]
        v = vb[mask] / denom
        w = vc[mask] / denom
        proj = a + ab * v[:, None] + ac * w[:, None]
        out[mask] = np.sum((p[mask] - proj) ** 2, axis=1)
    return out

def points_to_mesh_distance(points: np.ndarray, vertices: np.ndarray, faces: np.ndarray) -> np.ndarray:
    if len(points) == 0:
        return np.array([], dtype=np.float64)
    min_d2 = np.full((len(points),), np.inf, dtype=np.float64)
    for face in faces:
        min_d2 = np.minimum(min_d2, point_triangle_dist2(points, vertices[face]))
    return np.sqrt(min_d2)

def load_hand_collision_meshes() -> dict[str, dict[str, Any]]:
    model = mujoco.MjModel.from_xml_path(str(HAND_XML))
    meshes = {}
    for geom_id in range(model.ngeom):
        geom_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, geom_id) or ''
        if not geom_name.endswith('_collision'):
            continue
        body_id = int(model.geom_bodyid[geom_id])
        body_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, body_id) or ''
        mesh_id = int(model.geom_dataid[geom_id])
        vadr, vnum = int(model.mesh_vertadr[mesh_id]), int(model.mesh_vertnum[mesh_id])
        fadr, fnum = int(model.mesh_faceadr[mesh_id]), int(model.mesh_facenum[mesh_id])
        verts_geom = np.asarray(model.mesh_vert[vadr:vadr+vnum], dtype=np.float64)
        faces = np.asarray(model.mesh_face[fadr:fadr+fnum], dtype=np.int64)
        rot = quat_to_mat(np.asarray(model.geom_quat[geom_id], dtype=np.float64))
        pos = np.asarray(model.geom_pos[geom_id], dtype=np.float64)
        meshes[body_name] = {
            'geom_name': geom_name,
            'vertices': pos + verts_geom @ rot.T,
            'faces': faces,
        }
    return meshes

hand_meshes = load_hand_collision_meshes()
print('hand collision links:', len(hand_meshes))

def rpy_to_mat(rpy: np.ndarray) -> np.ndarray:
    roll, pitch, yaw = [float(v) for v in rpy]
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw), np.sin(yaw)
    rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]], dtype=np.float64)
    ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]], dtype=np.float64)
    rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]], dtype=np.float64)
    return rz @ ry @ rx

def axis_angle_to_mat(axis: np.ndarray, angle: float) -> np.ndarray:
    axis = np.asarray(axis, dtype=np.float64)
    norm = np.linalg.norm(axis)
    if norm < 1e-12:
        return np.eye(3, dtype=np.float64)
    x, y, z = axis / norm
    c, s = np.cos(float(angle)), np.sin(float(angle))
    C = 1.0 - c
    return np.array([
        [c + x*x*C, x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s, c + y*y*C, y*z*C - x*s],
        [z*x*C - y*s, z*y*C + x*s, c + z*z*C],
    ], dtype=np.float64)

def transform_matrix(xyz: np.ndarray | list[float] | None = None, rpy: np.ndarray | list[float] | None = None) -> np.ndarray:
    T = np.eye(4, dtype=np.float64)
    if rpy is not None:
        T[:3, :3] = rpy_to_mat(np.asarray(rpy, dtype=np.float64))
    if xyz is not None:
        T[:3, 3] = np.asarray(xyz, dtype=np.float64)
    return T

def parse_xyz(text: str | None, default: tuple[float, float, float] = (0.0, 0.0, 0.0)) -> np.ndarray:
    if not text:
        return np.asarray(default, dtype=np.float64)
    return np.asarray([float(v) for v in text.split()], dtype=np.float64)

def load_urdf_hand() -> dict[str, Any]:
    root = ET.parse(HAND_URDF).getroot()
    links: dict[str, dict[str, Any]] = {link.attrib['name']: {'visuals': []} for link in root.findall('link')}
    joints: dict[str, dict[str, Any]] = {}
    children_by_parent: dict[str, list[str]] = {}
    for link in root.findall('link'):
        link_name = link.attrib['name']
        for visual in link.findall('visual'):
            geom = visual.find('geometry')
            mesh = geom.find('mesh') if geom is not None else None
            if mesh is None or 'filename' not in mesh.attrib:
                continue
            origin = visual.find('origin')
            xyz = parse_xyz(origin.attrib.get('xyz') if origin is not None else None)
            rpy = parse_xyz(origin.attrib.get('rpy') if origin is not None else None)
            mesh_path = (HAND_URDF.parent / mesh.attrib['filename']).resolve()
            tri = trimesh.load_mesh(mesh_path, process=False)
            if not isinstance(tri, trimesh.Trimesh):
                tri = trimesh.util.concatenate(tuple(tri.geometry.values()))
            links[link_name]['visuals'].append({
                'mesh_path': mesh_path,
                'vertices': np.asarray(tri.vertices, dtype=np.float32),
                'faces': np.asarray(tri.faces, dtype=np.uint32),
                'origin': transform_matrix(xyz, rpy),
            })
    for joint in root.findall('joint'):
        name = joint.attrib['name']
        parent = joint.find('parent').attrib['link']
        child = joint.find('child').attrib['link']
        origin = joint.find('origin')
        xyz = parse_xyz(origin.attrib.get('xyz') if origin is not None else None)
        rpy = parse_xyz(origin.attrib.get('rpy') if origin is not None else None)
        axis_node = joint.find('axis')
        axis = parse_xyz(axis_node.attrib.get('xyz') if axis_node is not None else None, default=(1.0, 0.0, 0.0))
        joints[name] = {
            'name': name,
            'type': joint.attrib.get('type', 'fixed'),
            'parent': parent,
            'child': child,
            'origin': transform_matrix(xyz, rpy),
            'axis': axis,
        }
        children_by_parent.setdefault(parent, []).append(name)
    movable = [name for name, joint in joints.items() if joint['type'] in {'revolute', 'continuous', 'prismatic'}]
    base_order = ['ARTx', 'ARTy', 'ARTz', 'ARRx', 'ARRy', 'ARRz']
    finger_order = sorted(name for name in movable if name not in set(base_order))
    qpos_order = base_order + finger_order
    return {'links': links, 'joints': joints, 'children_by_parent': children_by_parent, 'qpos_order': qpos_order}

def joint_motion(joint: dict[str, Any], value: float) -> np.ndarray:
    T = np.eye(4, dtype=np.float64)
    if joint['type'] == 'prismatic':
        T[:3, 3] = joint['axis'] * float(value)
    elif joint['type'] in {'revolute', 'continuous'}:
        T[:3, :3] = axis_angle_to_mat(joint['axis'], float(value))
    return T

def urdf_fk(urdf: dict[str, Any], urdf_dof: np.ndarray) -> dict[str, np.ndarray]:
    values = {name: float(urdf_dof[idx]) for idx, name in enumerate(urdf['qpos_order'][:len(urdf_dof)])}
    transforms = {'floating_base': np.eye(4, dtype=np.float64)}
    stack = ['floating_base']
    while stack:
        parent = stack.pop()
        for joint_name in urdf['children_by_parent'].get(parent, []):
            joint = urdf['joints'][joint_name]
            child = joint['child']
            transforms[child] = transforms[parent] @ joint['origin'] @ joint_motion(joint, values.get(joint_name, 0.0))
            stack.append(child)
    return transforms

def transformed_urdf_meshes(urdf: dict[str, Any], payload: dict[str, Any], frame: int, offset: np.ndarray) -> list[dict[str, Any]]:
    urdf_dof = np.asarray(payload['hand_trajectory']['urdf_dof'][frame], dtype=np.float64)
    transforms = urdf_fk(urdf, urdf_dof)
    out = []
    for link_name, link in urdf['links'].items():
        link_T = transforms.get(link_name)
        if link_T is None:
            continue
        for visual_idx, visual in enumerate(link['visuals']):
            T = link_T @ visual['origin']
            vertices = np.asarray(visual['vertices'], dtype=np.float64)
            world_vertices = vertices @ T[:3, :3].T + T[:3, 3] + offset
            out.append({
                'link_name': link_name,
                'visual_idx': visual_idx,
                'vertices': world_vertices.astype(np.float32),
                'faces': visual['faces'],
            })
    return out

urdf_hand = load_urdf_hand()
print('URDF visual mesh links:', sum(1 for link in urdf_hand['links'].values() if link['visuals']))
print('URDF qpos order:', urdf_hand['qpos_order'])


In [ ]:
def flatten_contactbench(payload: dict[str, Any], backend: str) -> pd.DataFrame:
    rows = []
    for frame_idx, frame in enumerate(payload['contact']):
        for entry_idx, entry in enumerate(frame):
            link_name = entry['joint_name']
            pairs = entry.get('contact_pairs', [])
            for pair_idx, pair in enumerate(pairs):
                pos_world = np.asarray(pair['pos_world'], dtype=np.float64)
                pos_joint = np.asarray(pair['pos_joint'], dtype=np.float64)
                force_normal = np.asarray(pair['force_normal'], dtype=np.float64)
                rows.append({
                    'backend': backend,
                    'frame': int(frame_idx),
                    'entry_idx': int(entry_idx),
                    'pair_idx': int(pair_idx),
                    'link_name': link_name,
                    'object_name': entry['object_name'],
                    'pos_world': pos_world,
                    'pos_joint': pos_joint,
                    'normal_force_norm': float(np.linalg.norm(force_normal)),
                    'x': float(pos_world[0]),
                    'y': float(pos_world[1]),
                    'z': float(pos_world[2]),
                })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        df['distance_to_hand_mesh_mm'] = []
        return df
    df['distance_to_hand_mesh_mm'] = np.nan
    for link_name, group in df.groupby('link_name'):
        mesh = hand_meshes.get(link_name)
        if mesh is None:
            continue
        points = np.stack(group['pos_joint'].to_numpy())
        dist_mm = points_to_mesh_distance(points, mesh['vertices'], mesh['faces']) * 1000.0
        df.loc[group.index, 'distance_to_hand_mesh_mm'] = dist_mm
    return df

cpu_df = flatten_contactbench(cpu_payload, 'cpu_mujoco')
gpu_df = flatten_contactbench(gpu_payload, 'mjx_warp_gpu')
contact_df = pd.concat([cpu_df, gpu_df], ignore_index=True)
display(contact_df.groupby('backend')['distance_to_hand_mesh_mm'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))


In [ ]:
def object_positions(payload: dict[str, Any], frame: int) -> np.ndarray:
    return np.asarray([traj['pos'][frame] for traj in payload['object_trajectories']], dtype=np.float32)

def palm_positions(payload: dict[str, Any]) -> np.ndarray:
    return np.asarray(payload['hand_trajectory']['mano_global_pos'], dtype=np.float32)

def frame_contacts(df: pd.DataFrame, backend: str, frame: int) -> pd.DataFrame:
    return df[(df['backend'] == backend) & (df['frame'] == frame)]

def colors_from_error(errors_mm: np.ndarray, max_mm: float = 8.0, alpha: int = 255, lighten: float = 0.0) -> np.ndarray:
    norm = np.clip(errors_mm / max_mm, 0.0, 1.0)
    red = 255.0 * norm
    green = (190.0 * (1.0 - np.abs(norm - 0.45))).clip(40, 210)
    blue = 255.0 * (1.0 - norm)
    rgb = np.column_stack([red, green, blue])
    if lighten > 0.0:
        rgb = rgb * (1.0 - lighten) + 255.0 * lighten
    a = np.full(len(errors_mm), int(alpha), dtype=np.uint8)
    return np.column_stack([rgb.clip(0, 255).astype(np.uint8), a]).astype(np.uint8)

def stack_positions(series: pd.Series) -> np.ndarray:
    if len(series) == 0:
        return np.empty((0, 3), dtype=np.float32)
    return np.stack(series.to_numpy()).astype(np.float32)

def active_object_names(active: pd.DataFrame) -> set[str]:
    if len(active) == 0:
        return set()
    return set(active['object_name'].astype(str))

def summary_markdown() -> str:
    rows = []
    for label, backend in [('CPU MuJoCo', 'cpu_mujoco'), ('MJX-Warp GPU', 'mjx_warp_gpu')]:
        values = contact_df.loc[contact_df['backend'] == backend, 'distance_to_hand_mesh_mm'].dropna().to_numpy()
        pct = np.percentile(values, [50, 95, 99]) if len(values) else [np.nan, np.nan, np.nan]
        rows.append(
            f"| {label} | {len(values)} | {np.mean(values):.3f} | {pct[0]:.3f} | {pct[1]:.3f} | {pct[2]:.3f} | {np.max(values):.3f} |"
        )
    return '\n'.join([
        '# ContactBench CPU vs MJX-Warp GPU',
        '',
        'Error = raw native contact point distance to corresponding hand collision mesh surface. No projection is applied.',
        '',
        '| Backend | Count | Mean mm | Median mm | P95 mm | P99 mm | Max mm |',
        '|---|---:|---:|---:|---:|---:|---:|',
        *rows,
        '',
        'GPU force fields in ContactBench JSON are normal-force proxies from MJX-Warp efc rows, not CPU mj_contactForce 6D wrenches.',
    ])

print(summary_markdown())


## Log To Rerun

Default `FRAME_STRIDE = 5` logs every 5th frame for responsiveness. Set it to `1` for all 1000 frames.


In [ ]:
FRAME_STRIDE = 5
BALL_RADIUS = float(cpu_payload['metadata'].get('ball_radius', 0.03))
CONTACT_HISTORY_RADIUS = 0.002
ACTIVE_CONTACT_RADIUS = 0.008
OFFSETS = {
    'cpu_mujoco': np.array([-0.42, 0.0, 0.0], dtype=np.float32),
    'mjx_warp_gpu': np.array([0.42, 0.0, 0.0], dtype=np.float32),
}
PAYLOADS = {
    'cpu_mujoco': cpu_payload,
    'mjx_warp_gpu': gpu_payload,
}
BACKEND_LABELS = {
    'cpu_mujoco': 'CPU MuJoCo',
    'mjx_warp_gpu': 'MJX-Warp GPU',
}
BACKEND_LABEL_COLORS = {
    'cpu_mujoco': [80, 150, 255],
    'mjx_warp_gpu': [255, 170, 70],
}
HAND_MESH_COLORS = {
    'cpu_mujoco': np.array([95, 150, 255, 180], dtype=np.uint8),
    'mjx_warp_gpu': np.array([255, 178, 86, 180], dtype=np.uint8),
}
SHOW_INACTIVE_BALLS = False
SHOW_HISTORICAL_CONTACTS = False
BALL_INACTIVE_COLOR = np.array([245, 248, 255, 90], dtype=np.uint8)
BALL_ACTIVE_COLORS = {
    'cpu_mujoco': np.array([80, 170, 255, 255], dtype=np.uint8),
    'mjx_warp_gpu': np.array([255, 195, 70, 255], dtype=np.uint8),
}
CONTACT_HISTORY_ALPHA = 34
CONTACT_ACTIVE_ALPHA = 255

rr.init('contactbench_cpu_gpu_contacts', recording_id=uuid.uuid4(), spawn=False)
rr.log('world/origin', rr.Points3D([[0, 0, 0]], colors=[[255, 255, 255]], radii=0.018), static=True)

# Static all-contact clouds and palm trajectories.
for backend, payload in PAYLOADS.items():
    offset = OFFSETS[backend]
    label_pos = offset + np.array([-0.12, -0.26, 0.36], dtype=np.float32)
    rr.log(
        f'world/{backend}/label',
        rr.Points3D([label_pos], colors=[BACKEND_LABEL_COLORS[backend]], radii=0.02, labels=[BACKEND_LABELS[backend]], show_labels=True),
        static=True,
    )
    backend_df = contact_df[contact_df['backend'] == backend]
    all_positions = stack_positions(backend_df['pos_world']) + offset
    all_errors = backend_df['distance_to_hand_mesh_mm'].fillna(0).to_numpy()
    if SHOW_HISTORICAL_CONTACTS:
        rr.log(
            f'world/{backend}/all_contacts',
            rr.Points3D(
                all_positions,
                colors=colors_from_error(all_errors, alpha=CONTACT_HISTORY_ALPHA, lighten=0.65),
                radii=np.full(len(all_positions), CONTACT_HISTORY_RADIUS, dtype=np.float32),
            ),
            static=True,
        )
    else:
        rr.log(f'world/{backend}/all_contacts', rr.Clear(recursive=True), static=True)
    palm = palm_positions(payload) + offset
    rr.log(
        f'world/{backend}/palm_path',
        rr.LineStrips3D([palm], colors=[BACKEND_LABEL_COLORS[backend]], radii=0.003),
        static=True,
    )

# Animated frame-by-frame balls and active contacts.
frame_count = len(cpu_payload['contact'])
for frame in range(0, frame_count, FRAME_STRIDE):
    rr.set_time('frame', sequence=frame)
    for backend, payload in PAYLOADS.items():
        offset = OFFSETS[backend]
        active = frame_contacts(contact_df, backend, frame)
        balls = object_positions(payload, frame) + offset
        object_names = [traj['object_name'] for traj in payload['object_trajectories']]
        active_names = active_object_names(active)
        active_ball_mask = np.asarray([name in active_names for name in object_names], dtype=bool)
        inactive_balls = balls[~active_ball_mask]
        active_balls = balls[active_ball_mask]
        if SHOW_INACTIVE_BALLS:
            rr.log(
                f'world/{backend}/inactive_balls',
                rr.Points3D(
                    inactive_balls,
                    colors=np.tile(BALL_INACTIVE_COLOR, (len(inactive_balls), 1)),
                    radii=np.full(len(inactive_balls), BALL_RADIUS * 0.70, dtype=np.float32),
                ),
            )
        else:
            rr.log(f'world/{backend}/inactive_balls', rr.Clear(recursive=True))
        rr.log(
            f'world/{backend}/contacting_balls',
            rr.Points3D(
                active_balls,
                colors=np.tile(BALL_ACTIVE_COLORS[backend], (len(active_balls), 1)),
                radii=np.full(len(active_balls), BALL_RADIUS * 1.25, dtype=np.float32),
            ),
        )
        contact_positions = stack_positions(active['pos_world']) + offset
        contact_errors = active['distance_to_hand_mesh_mm'].fillna(0).to_numpy() if len(active) else np.array([], dtype=np.float64)
        contact_colors = colors_from_error(contact_errors, alpha=CONTACT_ACTIVE_ALPHA) if len(active) else np.empty((0, 4), dtype=np.uint8)
        rr.log(
            f'world/{backend}/active_contacts',
            rr.Points3D(
                contact_positions,
                colors=contact_colors,
                radii=np.full(len(contact_positions), ACTIVE_CONTACT_RADIUS, dtype=np.float32),
            ),
        )
        palm = palm_positions(payload)[frame] + offset
        rr.log(
            f'world/{backend}/palm_current',
            rr.Points3D([palm], colors=[BACKEND_LABEL_COLORS[backend]], radii=0.012),
        )
        for mesh in transformed_urdf_meshes(urdf_hand, payload, frame, offset):
            color = np.tile(HAND_MESH_COLORS[backend], (len(mesh['vertices']), 1))
            rr.log(
                f"world/{backend}/hand_urdf/{mesh['link_name']}_{mesh['visual_idx']}",
                rr.Mesh3D(
                    vertex_positions=mesh['vertices'],
                    triangle_indices=mesh['faces'],
                    vertex_colors=color,
                ),
            )

print(f'Logged frames 0..{frame_count - 1} with stride={FRAME_STRIDE} to Rerun, including animated URDF hand meshes')


In [ ]:
# Increase height if the embedded viewer appears too small in VS Code/Jupyter.
rr.notebook_show(width=2500, height=1200)
